# Блокнот 3. Линейная регрессия: своими руками и в scikit-learn

Курс «Машинное обучение с нуля», модуль 3. Разделы блокнота идут за уроками: 1 — урок 3.1, 2 — 3.3, 3 — 3.4, 4 — 3.5, 5 — 3.6, 6–7 — 3.7–3.8. Выполняйте ячейки сверху вниз (**Shift + Enter**). Там, где `...  # ваш код`, допишите решение; ячейка с `assert` после него проверит ответ.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(42)

## 1. Векторы, матрицы и предсказания (урок 3.1)

Три квартиры на Корусанте: столбцы — площадь (м²) и этаж. Модель: цена = 12·площадь − 5·этаж + 100.

**Задание 1.** Посчитайте предсказания для всех трёх квартир сразу — одним матричным умножением `X @ w` и прибавлением `b`.

In [ ]:
X = np.array([[50.0, 3.0],
              [80.0, 10.0],
              [30.0, 1.0]])
w = np.array([12.0, -5.0])
b = 100.0
y_hat = ...  # ваш код
y_hat

In [ ]:
assert np.allclose(y_hat, [685, 1010, 455]), f"получилось {y_hat}"
print("Верно!")

## 2. Производная и градиент (урок 3.3)

Производную можно проверить численно: $f'(x) \approx \frac{f(x + h) - f(x - h)}{2h}$ при маленьком $h$.

In [ ]:
def numeric_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)


f = lambda x: 3 * x ** 2 + 2 * x  # производная: 6x + 2
print(numeric_derivative(f, 1.0), 6 * 1.0 + 2)

**Задание 2.** Напишите градиент MSE для модели $\hat{y} = Xw + b$:
$\frac{\partial L}{\partial w} = -\frac{2}{n} X^T (y - \hat{y})$, $\frac{\partial L}{\partial b} = -\frac{2}{n} \sum (y - \hat{y})$.

In [ ]:
def mse(X, y, w, b):
    return np.mean((y - (X @ w + b)) ** 2)


def mse_grad(X, y, w, b):
    grad_w = ...  # ваш код
    grad_b = ...  # ваш код
    return grad_w, grad_b

In [ ]:
Xg = rng.normal(size=(20, 3))
yg = rng.normal(size=20)
wg, bg = rng.normal(size=3), 0.5
gw, gb = mse_grad(Xg, yg, wg, bg)
num_w = [numeric_derivative(lambda t: mse(Xg, yg, np.where(np.arange(3) == j, t, wg), bg), wg[j]) for j in range(3)]
num_b = numeric_derivative(lambda t: mse(Xg, yg, wg, t), bg)
assert np.allclose(gw, num_w, atol=1e-6) and np.isclose(gb, num_b, atol=1e-6), "градиент не совпал с численным"
print("Верно! Аналитический градиент совпал с численным.")

## 3. Метод наименьших квадратов (урок 3.4)

Сгенерируем данные с известными весами и проверим, что формула их находит. Столбец единиц добавляем, чтобы свободный член $b$ стал обычным весом.

**Задание 3.** Решите нормальное уравнение $X^T X w = X^T y$ функцией `np.linalg.solve` (обратную матрицу явно не считаем).

In [ ]:
n = 200
Xs = rng.normal(size=(n, 2))
ys = 3 * Xs[:, 0] - 2 * Xs[:, 1] + 5 + rng.normal(0, 0.1, n)
X1 = np.c_[np.ones(n), Xs]   # первый столбец — единицы
w_ols = ...  # ваш код
w_ols.round(3)

In [ ]:
assert np.allclose(w_ols, [5, 3, -2], atol=0.05), f"получилось {w_ols}"
from sklearn.linear_model import LinearRegression

sk = LinearRegression().fit(Xs, ys)
print("Верно! scikit-learn:", round(sk.intercept_, 3), sk.coef_.round(3))

## 4. Градиентный спуск своими руками (урок 3.5)

**Задание 4.** Допишите шаг спуска: веса сдвигаются против градиента, умноженного на скорость обучения `lr`.

In [ ]:
def gradient_descent(X, y, lr=0.1, steps=200):
    w, b = np.zeros(X.shape[1]), 0.0
    history = []
    for _ in range(steps):
        gw, gb = mse_grad(X, y, w, b)
        w = ...  # ваш код
        b = ...  # ваш код
        history.append(mse(X, y, w, b))
    return w, b, history


w_gd, b_gd, hist = gradient_descent(Xs, ys)
print(w_gd.round(3), round(b_gd, 3))

In [ ]:
assert np.allclose(w_gd, [3, -2], atol=0.05) and abs(b_gd - 5) < 0.05, "веса не сошлись к 3, −2 и 5"
plt.plot(hist)
plt.yscale("log")
plt.xlabel("шаг")
plt.ylabel("MSE")
plt.title("Ошибка падает с каждым шагом")
plt.show()

Поэкспериментируйте: запустите `gradient_descent(Xs, ys, lr=1.2)` — что случилось с ошибкой? А с `lr=0.001`?

## 5. Метрики регрессии своими руками (урок 3.6)

**Задание 5.** Посчитайте $R^2 = 1 - \frac{\sum (y - \hat{y})^2}{\sum (y - \bar{y})^2}$ без библиотек и сравните с `r2_score`.

In [ ]:
from sklearn.metrics import r2_score

y_true = np.array([3.0, 5.0, 7.0, 9.0])
y_pred = np.array([2.5, 5.5, 6.0, 9.5])
r2 = ...  # ваш код

In [ ]:
assert np.isclose(r2, r2_score(y_true, y_pred)), f"получилось {r2}, а r2_score даёт {r2_score(y_true, y_pred)}"
print("Верно!", round(r2, 4))

## 6. Настоящие данные: цены домов в Калифорнии (урок 3.8)

20 640 районов по переписи 1990 года, 8 признаков, целевая — медианная стоимость дома в сотнях тысяч долларов. scikit-learn скачает набор из интернета.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = fetch_california_housing(as_frame=True)
df = data.frame
df.describe().round(2)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

baseline = DummyRegressor()
print("baseline R² (CV):", cross_val_score(baseline, X_train, y_train, cv=5, scoring="r2").mean().round(3))

**Задание 6.** Соберите пайплайн из `StandardScaler()` и `LinearRegression()` (функция `make_pipeline`), оцените его 5-кратной кросс-валидацией по метрике `"r2"` и сохраните среднее в `cv_r2`.

In [ ]:
model = ...  # ваш код
cv_r2 = ...  # ваш код
print(round(cv_r2, 3))

In [ ]:
assert 0.58 < cv_r2 < 0.64, f"получилось {cv_r2}"
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("тест: R²", round(r2_score(y_test, pred), 3), " RMSE", round(mean_squared_error(y_test, pred) ** 0.5, 3),
      " MAE", round(mean_absolute_error(y_test, pred), 3))
dict(zip(data.feature_names, model[-1].coef_.round(3)))

In [ ]:
plt.scatter(pred, y_test - pred, s=3, alpha=0.3)
plt.axhline(0, color="orange")
plt.xlabel("предсказание")
plt.ylabel("остаток y − ŷ")
plt.title("Наклонная полоса — районы с ценой, обрезанной на 5,0")
plt.show()

## 7. Полиномиальные признаки и регуляризация (уроки 3.7–3.8)

Многочлен второй степени из 8 признаков даёт 45 признаков. Сравним кросс-валидацию — обратите внимание на разброс оценок по фолдам.

In [ ]:
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import PolynomialFeatures

for name, m in [("линейная", make_pipeline(StandardScaler(), LinearRegression())),
                ("многочлен 2", make_pipeline(PolynomialFeatures(2), StandardScaler(), LinearRegression())),
                ("многочлен 2 + Ridge", make_pipeline(PolynomialFeatures(2), StandardScaler(), Ridge(alpha=10)))]:
    scores = cross_val_score(m, X_train, y_train, cv=5, scoring="r2")
    print(f"{name:22s} R² по фолдам: {np.round(scores, 2)}")

В одном из фолдов многочлен даёт огромную ошибку: в данных есть районы с экстремальными значениями `AveOccup` и `AveRooms` (максимум `AveOccup` — 1243), а квадраты этих чисел ещё больше. Модель, которая на тесте выглядит лучше, на кросс-валидации оказывается ненадёжной — это и есть причина не верить одной оценке.

**Lasso** обнуляет веса неважных признаков:

In [ ]:
lasso = make_pipeline(StandardScaler(), Lasso(alpha=0.1)).fit(X_train, y_train)
dict(zip(data.feature_names, lasso[-1].coef_.round(3)))